# Lab 3 — AST-Aware Editing

**Goal:** Before asking the LLM to fix code, give it a structural understanding via Python's `ast` module.  
This prevents the agent from accidentally breaking working functions while fixing a broken one.

**Why AST matters:**  
A naive agent gets the whole file and may rewrite it entirely.  
An AST-aware agent knows: 'only `calculate_discount` is broken — leave the other 12 functions alone.'

In [ ]:
import ast
import sys, os
sys.path.insert(0, os.path.dirname(os.path.abspath('__file__')))
from dotenv import load_dotenv
load_dotenv(override=True)
from openai import OpenAI
from sandbox import run_pytest

client = OpenAI()
print('Ready ✓')

## 1. Parse and inspect an AST

In [ ]:
SOURCE = """
class ShoppingCart:
    def __init__(self):
        self.items = []

    def add_item(self, name: str, price: float, qty: int = 1):
        self.items.append({'name': name, 'price': price, 'qty': qty})

    def total(self) -> float:
        return sum(i['price'] * i['qty'] for i in self.items)

    def calculate_discount(self, pct: float) -> float:
        # BUG: should apply discount to total, not return pct directly
        return pct

    def item_count(self) -> int:
        return len(self.items)
"""

tree = ast.parse(SOURCE)

for node in ast.walk(tree):
    if isinstance(node, ast.FunctionDef):
        args = [a.arg for a in node.args.args]
        print(f'  def {node.name}({', '.join(args)}) at line {node.lineno}')
    elif isinstance(node, ast.ClassDef):
        print(f'class {node.name} at line {node.lineno}')

## 2. Surgical fix — only change the broken function

In [ ]:
def extract_function_source(source: str, func_name: str) -> str:
    """Extract a single function's source lines from a file."""
    tree = ast.parse(source)
    lines = source.splitlines()
    for node in ast.walk(tree):
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)) and node.name == func_name:
            return '\n'.join(lines[node.lineno - 1 : node.end_lineno])
    return ''

broken_func = extract_function_source(SOURCE, 'calculate_discount')
print('Broken function:')
print(broken_func)

In [ ]:
TESTS = """
def test_discount():
    cart = ShoppingCart()
    cart.add_item('apple', 2.00, 3)
    assert cart.calculate_discount(10) == 5.40  # 10% off 6.00

def test_total_unchanged():
    cart = ShoppingCart()
    cart.add_item('book', 10.00)
    assert cart.total() == 10.00
    assert cart.item_count() == 1
"""

prompt = f"""Fix ONLY the function `calculate_discount` in this class.
Do not change any other method.

Full class source:
```python
{SOURCE}
```

The broken function is:
```python
{broken_func}
```

Tests that must pass:
{TESTS}

Output the complete fixed class source. No fences."""

response = client.chat.completions.create(
    model='gpt-4o-mini',
    messages=[
        {'role': 'system', 'content': 'You are an expert Python engineer. Output only raw Python code.'},
        {'role': 'user', 'content': prompt},
    ],
    temperature=0.1,
)
fixed = response.choices[0].message.content
print('Fixed source:')
print(fixed)

In [ ]:
result = run_pytest(TESTS, fixed)
print(result.summary())

## 3. Verify other functions were not broken

In [ ]:
original_methods = {}
fixed_methods = {}

for src, store in [(SOURCE, original_methods), (fixed, fixed_methods)]:
    for node in ast.walk(ast.parse(src)):
        if isinstance(node, ast.FunctionDef):
            store[node.name] = node.lineno

print('Methods in original:', list(original_methods.keys()))
print('Methods in fixed:   ', list(fixed_methods.keys()))
print('Same methods?', set(original_methods) == set(fixed_methods))

## Exercise
Build a function `ast_diff(source_a, source_b)` that returns which function names were added, removed, or modified between two versions of a Python file.  
**Hint:** Compare `ast.dump(node)` for each function.

In [ ]:
# Your code here
